# Veritas — Model Evaluation & EDA

This notebook walks through:
1. Dataset exploration (WELFake)
2. TF-IDF model training & evaluation
3. Confusion matrix & ROC curve
4. Feature importance analysis
5. Error analysis on misclassified examples

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
ACCENT = '#4f8cff'
REAL_COLOR = '#00e5a0'
FAKE_COLOR = '#ff4757'

print('Libraries loaded ✓')

## 1. Load Dataset

In [ ]:
# Load WELFake dataset
# Download from: https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification
DATA_PATH = Path('../data/WELFake_Dataset.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
    df = df.dropna(subset=['text'])
    print(f'Loaded {len(df):,} articles')
    print(f'Real: {(df.label==1).sum():,} | Fake: {(df.label==0).sum():,}')
    df.head()
else:
    print('⚠ Dataset not found at', DATA_PATH)
    print('Using built-in demo corpus instead...')
    import sys; sys.path.insert(0, '../..')
    # Use built-in corpus from model_service

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
counts = df['label'].value_counts()
axes[0].bar(['Fake', 'Real'], [counts[0], counts[1]], color=[FAKE_COLOR, REAL_COLOR], alpha=0.85)
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')

# Text length distribution
df['text_len'] = df['combined'].str.split().str.len()
axes[1].hist(df[df.label==1]['text_len'].clip(0,1000), bins=50, color=REAL_COLOR, alpha=0.6, label='Real')
axes[1].hist(df[df.label==0]['text_len'].clip(0,1000), bins=50, color=FAKE_COLOR, alpha=0.6, label='Fake')
axes[1].set_title('Article Length Distribution', fontsize=13)
axes[1].set_xlabel('Word count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Train & Evaluate TF-IDF Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

texts = df['combined'].tolist()
labels = df['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=100000, ngram_range=(1, 3),
    sublinear_tf=True, min_df=2, stop_words='english'
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# PAC classifier
clf = PassiveAggressiveClassifier(max_iter=1000, C=0.5, random_state=42)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

## 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Fake', 'Pred Real'],
            yticklabels=['True Fake', 'True Real'], ax=ax)
ax.set_title('Confusion Matrix — TF-IDF + PAC', fontsize=13)
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. ROC Curve

In [ ]:
decisions = clf.decision_function(X_test_vec)
fpr, tpr, _ = roc_curve(y_test, decisions)
auc = roc_auc_score(y_test, decisions)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color=ACCENT, lw=2, label=f'ROC AUC = {auc:.4f}')
plt.plot([0,1],[0,1],'--', color='gray', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — TF-IDF + PAC')
plt.legend()
plt.tight_layout()
plt.savefig('../data/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Top TF-IDF Features for Fake vs Real

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

# Top features for REAL (positive coef) and FAKE (negative coef)
top_real_idx = np.argsort(coefs)[-20:]
top_fake_idx = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(feature_names[top_real_idx], coefs[top_real_idx], color=REAL_COLOR, alpha=0.8)
axes[0].set_title('Top 20 Real News Indicators', fontsize=12)
axes[0].set_xlabel('Coefficient weight')

axes[1].barh(feature_names[top_fake_idx], np.abs(coefs[top_fake_idx]), color=FAKE_COLOR, alpha=0.8)
axes[1].set_title('Top 20 Fake News Indicators', fontsize=12)
axes[1].set_xlabel('|Coefficient weight|')

plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()